In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report

from data_loading_code import load_data

In [23]:
class MLPClassifier(nn.Module):
    """Simple feed-forward neural network for binary text classification."""

    def __init__(self, input_dim: int, hidden_dims: list, num_classes: int = 2, dropout: float = 0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [24]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0
    all_preds, all_labels = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.cpu().tolist())
    n = len(loader.dataset)
    return total_loss / n, correct / n, all_preds, all_labels

In [25]:
HIDDEN_DIMS  = [256, 128]
DROPOUT      = 0.3
BATCH_SIZE   = 64
EPOCHS       = 5
LR           = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_train, y_train, X_val, y_val, vocab_size = load_data()
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Vocab: {vocab_size}")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=BATCH_SIZE)

Using device: cpu
Train: torch.Size([900, 7277]) | Val: torch.Size([100, 7277]) | Vocab: 7277


In [26]:
input_dim = X_train.shape[1]
model     = MLPClassifier(input_dim, HIDDEN_DIMS, dropout=DROPOUT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(model)

MLPClassifier(
  (net): Sequential(
    (0): Linear(in_features=7277, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [27]:
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}")

Epoch 01/5  train_loss=0.6911  train_acc=0.5433  val_loss=0.6831  val_acc=0.7100
Epoch 02/5  train_loss=0.6518  train_acc=0.9411  val_loss=0.6168  val_acc=0.8200
Epoch 03/5  train_loss=0.4705  train_acc=0.9956  val_loss=0.4464  val_acc=0.8500
Epoch 04/5  train_loss=0.1570  train_acc=0.9989  val_loss=0.3247  val_acc=0.8400
Epoch 05/5  train_loss=0.0219  train_acc=1.0000  val_loss=0.3241  val_acc=0.8500


In [28]:
_, _, preds, labels = evaluate(model, val_loader, criterion, device)
print("\nClassification Report (Validation):")
print(classification_report(labels, preds, target_names=["Negative", "Positive"]))


Classification Report (Validation):
              precision    recall  f1-score   support

    Negative       0.82      0.87      0.85        47
    Positive       0.88      0.83      0.85        53

    accuracy                           0.85       100
   macro avg       0.85      0.85      0.85       100
weighted avg       0.85      0.85      0.85       100

